In [65]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import mlflow
from mlflow.models import infer_signature
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe

import warnings
warnings.filterwarnings("ignore")

In [66]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
)

In [67]:
data.head
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


In [68]:
# ========================== Data Splitting into train test split ==========================
train, test = train_test_split(data, test_size=0.25, random_state=42)

X = train.drop(['quality'], axis=1).values.astype(np.float32)
y = train['quality'].values.astype(np.float32)

# Split train into train + validation
train_x, valid_x, train_y, valid_y = train_test_split(X, y, test_size=0.20, random_state=42)

test_x = test.drop(['quality'], axis=1).values.astype(np.float32)
test_y = test['quality'].values.astype(np.float32)

In [69]:

mean = np.mean(train_x, axis=-1)
mean.shape
print(mean)
print((f"Train y shape: {train_y.shape}, test y  shape {test_y.shape}"),)
print(f"Train shape: {train_x.shape}, Valid shape: {valid_x.shape}")


[15.795945 14.500731 16.484428 ... 12.976302 17.943855 22.38198 ]
Train y shape: (2938,), test y  shape (1225,)
Train shape: (2938, 11), Valid shape: (735, 11)


In [70]:
# Data normalization 
#1 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_x_scaled = scaler.fit_transform(train_x)   # Fit on train only
valid_x_scaled = scaler.transform(valid_x)
test_x_scaled  = scaler.transform(test_x)

In [75]:
# Infer signature for MLflow
signature = infer_signature(train_x, train_y)

print(f"Train shape: {train_x.shape}, Valid shape: {valid_x.shape}")



class InputNormalization(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        # Register as buffers so they move with the model to GPU and are saved
        self.register_buffer('mean', torch.tensor(mean, dtype=torch.float32))
        self.register_buffer('std',  torch.tensor(std,  dtype=torch.float32))
    
    def forward(self, x):
        return (x - self.mean) / (self.std + 1e-8)   # Avoid division by zero


# ========================== PyTorch Model ==========================
class WineQualityModel(nn.Module,):
    def __init__(self, input_dim,  mean=None, std=None):
        super().__init__()
        if mean is not None and std is not None:
            self.norm = InputNormalization(mean, std)
        else:
            self.norm = nn.Identity()   # No normalization
            
        self.model = nn.Sequential(
            
            nn.Linear(input_dim, 64),
        
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        x = self.norm(x)
        return self.model(x)

Train shape: (2938, 11), Valid shape: (735, 11)


In [76]:

print(torch.tensor(train_y).unsqueeze(-1).shape)
torch.tensor(train_y).shape
train_x.shape[1]

torch.Size([2938, 1])


11

In [77]:
def train_model(params, epochs, train_x, train_y, valid_x, valid_y, device):
    """
    Train the model with given hyperparameters
    """
    # Convert to tensors
    train_dataset = TensorDataset(torch.tensor(train_x), torch.tensor(train_y).unsqueeze(1))
    valid_dataset = TensorDataset(torch.tensor(valid_x), torch.tensor(valid_y).unsqueeze(1))
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)

    # Model, Loss, Optimizer
    model = WineQualityModel(input_dim=train_x.shape[1]).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.SGD(
        model.parameters(),
        lr=params["lr"],
        momentum=params["momentum"]
    )

    # Training Loop
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            # Validation
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch_x, batch_y in valid_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    outputs = model(batch_x)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()
            
            val_rmse = np.sqrt(val_loss / len(valid_loader))
            
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{epochs} - Val RMSE: {val_rmse:.4f}")

        # Final evaluation
        eval_rmse = val_rmse
        mlflow.log_metric("eval_rmse", eval_rmse)
        
        # Log the model
        mlflow.pytorch.log_model(model, "model", signature=signature)
        
        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [78]:
# ========================== Hyperparameter Search Space ==========================
space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0)
}


# ========================== Objective Function for Hyperopt ==========================
def objective(params):
    # You can change epochs as needed
    epochs = 50
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    result = train_model(params, epochs, train_x, train_y, valid_x, valid_y, device)
    return result

In [79]:
# ========================== Run Hyperparameter Tuning ==========================
mlflow.set_experiment("wine-quality-pytorch")

with mlflow.start_run(run_name="hyperopt_tuning"):
    trials = Trials()
    
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,           # Increased a bit from 4
        trials=trials
    )
    
    # Get best trial
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]
    
    mlflow.log_params(best)
    mlflow.log_metric("best_eval_rmse", best_run["loss"])
    
    print("\n" + "="*50)
    print(f"Best parameters: {best}")
    print(f"Best eval RMSE: {best_run['loss']:.4f}")
    print("="*50)

2026/05/01 16:58:37 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality-pytorch' does not exist. Creating a new experiment.


Epoch 1/50 - Val RMSE: 5339950519.9072               
Epoch 10/50 - Val RMSE: 0.8884                       
Epoch 20/50 - Val RMSE: 0.8888                       
Epoch 30/50 - Val RMSE: 0.8887                       
Epoch 40/50 - Val RMSE: 0.8888                       
Epoch 50/50 - Val RMSE: 0.8887                       
  0%|          | 0/8 [00:02<?, ?trial/s, best loss=?]

2026/05/01 16:58:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2026/05/01 16:58:40 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.



Epoch 1/50 - Val RMSE: 10.7762                                                 
Epoch 10/50 - Val RMSE: 0.8889                                                 
Epoch 20/50 - Val RMSE: 0.8886                                                 
Epoch 30/50 - Val RMSE: 0.8886                                                 
Epoch 40/50 - Val RMSE: 0.8885                                                 
Epoch 50/50 - Val RMSE: 0.8886                                                 
 12%|█▎        | 1/8 [00:07<00:38,  5.44s/trial, best loss: 0.8886626931299533]

2026/05/01 16:58:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2026/05/01 16:58:45 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.



...

Epoch 1/50 - Val RMSE: 2.3106                                                  
Epoch 10/50 - Val RMSE: 1.4976                                                 
Epoch 20/50 - Val RMSE: 1.1975                                                 
Epoch 30/50 - Val RMSE: 1.0505                                                 
Epoch 40/50 - Val RMSE: 0.9519                                                 
Epoch 50/50 - Val RMSE: 0.8917                                                 
 75%|███████▌  | 6/8 [00:27<00:08,  4.23s/trial, best loss: 0.7935440876470563]

2026/05/01 16:59:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2026/05/01 16:59:06 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.



Epoch 1/50 - Val RMSE: 30411094796379064.0000                                  
Epoch 10/50 - Val RMSE: 0.9313                                                 
Epoch 20/50 - Val RMSE: 0.8917                                                 
Epoch 30/50 - Val RMSE: 0.8891                                                 
Epoch 40/50 - Val RMSE: 0.8928                                                 
Epoch 50/50 - Val RMSE: 0.8930                                                 
 88%|████████▊ | 7/8 [00:32<00:04,  4.20s/trial, best loss: 0.7935440876470563]

2026/05/01 16:59:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.

2026/05/01 16:59:10 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.



100%|██████████| 8/8 [00:34<00:00,  4.34s/trial, best loss: 0.7935440876470563]

Best parameters: {'lr': np.float64(1.4822733408968838e-05), 'momentum': np.float64(0.9264303805060742)}
Best eval RMSE: 0.7935


In [81]:
print("\n" + "="*50)
print(f"Best parameters: {best}")
print(f"Best eval RMSE: {best_run['loss']:.4f}")
print("="*50)


Best parameters: {'lr': np.float64(1.4822733408968838e-05), 'momentum': np.float64(0.9264303805060742)}
Best eval RMSE: 0.7935


In [ ]:
## Inferencing

from mlflow.models import validate_serving_input

model_uri = 'runs:/9e0b52639ab44e6a864f5bd2f460fe42/model'

# The logged model does not contain an input_example.
# Manually generate a serving payload to verify your model prior to deployment.
from mlflow.models import convert_input_example_to_serving_input

# Define INPUT_EXAMPLE via assignment with your own input example to the model
# A valid input example is a data instance suitable for pyfunc prediction
serving_payload = convert_input_example_to_serving_input(test_x)

# Validate the serving payload works on the model
validate_serving_input(model_uri, serving_payload)

In [ ]:
mlflow.register_model(model_uri,"wine-quality")